# Desenvolvimento de módulo [Python](https://www.python.org/) para n-body gravitacional com energia;

Autor: Raphael Figueiredo Secchin

Data: 24/07/2026

---
## <font color="#1C77C3" >Informações de Sistema</font>

Nome do sistema operacional, nome do computador na rede, versão do kernel, arquitetura de hardware (32/64 bits), etc :

In [103]:
!uname -a

Linux a065fed08f91 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


Nome e versão do sistema operacional :

In [104]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


Diversas informações da CPU, como nome do processador, frequência em MHz da CPU, número de núcleos/cores, número de threads, memória cache, arquitetura de hardware (32/64 bits), etc :

In [105]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4399.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

Partições do sistema de arquivos do sistema operacional :

In [106]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
tmpfs           6.4G  3.5M  6.4G   1% /var/colab
/dev/sda1       114G   21G   93G  19% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


Memória RAM total e em uso pelo sistema operacional :

In [107]:
!free

               total        used        free      shared  buff/cache   available
Mem:        13286944     1202524     8296420        5568     3788000    11753200
Swap:              0           0           0


Versão do compilador C/C++ gcc :

In [108]:
!gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



O módulo Pyton "platform" fornece diversas informações do sistema (computador, sistema operacional, Python, etc).

In [109]:
import platform

In [110]:
platform.platform()

'Linux-6.6.122+-x86_64-with-glibc2.35'

Mais detalhes, como nome do computador na rede (node), arquitetura de hardware (32/64 bits), etc:

In [111]:
platform.uname()

uname_result(system='Linux', node='a065fed08f91', release='6.6.122+', version='#1 SMP Thu Apr 30 18:17:14 UTC 2026', machine='x86_64')

### Informações sobre Python e módulos

#### Python

Número da versão de Python:

In [112]:
platform.python_version()

'3.12.13'

data da versão:

In [113]:
platform.python_build()

('main', 'Mar  4 2026 09:23:07')

compilador C/C++ utilizado para criar tal versão de Python :

In [114]:
platform.python_compiler()

'GCC 11.4.0'

In [115]:
import pandas as pd

In [116]:
pd.__version__

'2.2.2'

In [117]:
import numba as nb

In [118]:
nb.__version__

'0.60.0'

In [119]:
import numpy as np

In [120]:
np.__version__

'2.0.2'

---
## <font color="#5EAAE8">Inicialização do Problema</font>

In [121]:
!pip install astroquery

In [122]:
from numba import njit
from astroquery.jplhorizons import Horizons
import numpy as np
import pandas as pd
import math

In [123]:
espacamento = 10
passos = 5000
g = 4 * np.pi**2 # em AU/ano
epsilon = 0.001
dt = 0.001
numCorpos = 9
frequenciaSnapshots = 20

A função calculaForcas calcula o valor das forças do sistema utilizando a fórmula:

$$ F_i = ∑_{\substack{j = 1 \\ j \neq i}}^{N} \frac{G m_{i} m_{j}}{|\vec{r}_{ij}|^{2}}$$

A função calculaEnergiaK calcula o valor da energia cinética utilizando a fórmula:

$$ K = ∑^{N}_{i = 1}\frac{m_{i}|\vec{v}_{i}|^{2}}{2}$$

A função calculaEnergiaU calcula o valor da energia cinética utilizando a fórmula:

$$ U = ∑^{N}_{i = 1}∑^{N}_{\substack{j = 1 \\ j \neq i}}\frac{G m_{i} m_{j}}{|r_{ij}|}$$

---
## <font color="#1B5A9D">Leitura dos dados</font>

In [124]:
EPOCH1 = '2026-jan-01'
EPOCH2 = '2026-jan-02'

#Ids dos corpos no Horizons
BODIES = {
    'Sol': 10,
    'Mercúrio': 199,
    'Vênus': 299,
    'Terra': 399,
    'Marte': 499,
    'Júpiter': 599,
    'Saturno': 699,
    'Urano': 799,
    'Netuno': 899
}

#Massas relativas ao Sol
MASSAS_SOLARES = {
    'Sol': 1.0,
    'Mercúrio': 1.660e-7,
    'Vênus': 2.447e-6,
    'Terra': 3.003e-6,
    'Marte': 3.227e-7,
    'Júpiter': 9.546e-4,
    'Saturno': 2.858e-4,
    'Urano': 4.366e-5,
    'Netuno': 5.151e-5
}

In [125]:
#Escrita do arquivo HDF5

posicoes = [] # em AU
velocidades = [] # em AU/ano
massas = []
nomes = []

for nome, objId in BODIES.items():
  obj = Horizons(id=objId, location='500@10', epochs = {'start': EPOCH1, 'stop': EPOCH2, 'step': '1d'})
  vec = obj.vectors()

  #Para as posições em AU
  pos = np.array([vec['x'][0], vec['y'][0], vec['z'][0]])

  #Para a velocidade
  vel = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * 365.25

  posicoes.append(pos)
  velocidades.append(vel)
  massas.append(MASSAS_SOLARES[nome])
  nomes.append(nome)

posicoes = np.array(posicoes)
velocidades = np.array(velocidades)
massas = np.array(massas)

df = pd.DataFrame({
    'x': posicoes[:, 0],
    'y': posicoes[:, 1],
    'z': posicoes[:, 2],
    'vx': velocidades[:, 0],
    'vy': velocidades[:, 1],
    'vz': velocidades[:, 2],
    'massa': massas
})

df['nome'] = nomes

df.to_hdf("Entrada.h5", key="Entrada", mode="w", format='table')

In [126]:
dados = pd.read_hdf("Entrada.h5")

---
## <font color="#4680AF">Funções de Snapshot</font>

In [127]:
def salvarHDF5(kInicial, uInicial, kFinal, uFinal, biblioteca, dimensao):
  df = pd.DataFrame({"EnergiaCineticaInicial":[kInicial], "EnergiaPotencialInicial":[uInicial], "EnergiaCineticaFinal":[kFinal], "EnergiaPotencialFinal":[uFinal]})
  df.to_hdf("SaidaTeste.h5", key=biblioteca + "/" + dimensao + "/Resultados", mode="a", append=True)

In [128]:
def salvarSnapshot(df, biblioteca, dimensao):
  df.to_hdf("SnapshotTeste.h5", key=biblioteca + "/" + dimensao + "/Snapshots", mode="a", append=True, format='table' )

---
## <font color="#33AAFF">Numba CPU</font>

---
### <font color="#E86C4A">3D</font>

In [130]:
x = dados["x"].to_numpy().copy()
y = dados["y"].to_numpy().copy()
z = dados["z"].to_numpy().copy()
vx = dados["vx"].to_numpy().copy()
vy = dados["vy"].to_numpy().copy()
vz = dados["vz"].to_numpy().copy()
m = dados["massa"].to_numpy().copy()
xTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

In [131]:
x = np.array(x, dtype=np.float64)
y = np.array(y, dtype=np.float64)
z = np.array(z, dtype=np.float64)
m = np.array(m, dtype=np.float64)
vx = np.array(vx, dtype=np.float64)
vy = np.array(vy, dtype=np.float64)
vz = np.array(vz, dtype=np.float64)

forcaX = np.array(forcaX, dtype=np.float64)
forcaY = np.array(forcaY, dtype=np.float64)
forcaZ = np.array(forcaZ, dtype=np.float64)

In [132]:
@njit(cache=True)
def calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy
      forcaZ[j] += -forcaAtual * dz

In [133]:
@njit(cache=True)
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ):
    for k in range(numCorpos):
      forcaX[k] = 0.0
      forcaY[k] = 0.0
      forcaZ[k] = 0.0

    calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ)
    for j in range(numCorpos):

      acelX = forcaX[j] / m[j]
      acelY = forcaY[j] / m[j]
      acelZ = forcaZ[j] / m[j]

      vx[j] = vx[j] + acelX * dt
      vy[j] = vy[j] + acelY * dt
      vz[j] = vz[j] + acelZ * dt

      x[j] = x[j] + vx[j] * dt
      y[j] = y[j] + vy[j] * dt
      z[j] = z[j] + vz[j] * dt

In [134]:
@njit(cache=True)
def calculaEnergiaK(m, vx, vy, vz, numCorpos):
  k = 0.0
  for i in range(numCorpos):
    k += 0.5 * m[i] * (vx[i]*vx[i] + vy[i]*vy[i] + vz[i]*vz[i])
  return k

In [135]:
@njit(cache=True)
def calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon):
  u = 0.0
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)**0.5

      u += - (g * m[i] * m[j]) / dist

  return u

Tempo de Execução:

In [136]:
%time kInicial = calculaEnergiaK(m, vx, vy, vz, numCorpos)

CPU times: user 1.3 s, sys: 125 ms, total: 1.43 s
Wall time: 1.59 s


In [137]:
%time uInicial = calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon)

CPU times: user 219 ms, sys: 3.76 ms, total: 223 ms
Wall time: 227 ms


In [138]:
print("Energia Mecânica Inicial: ", kInicial + uInicial)

Energia Mecânica inicial:  -0.00442745536003452


In [139]:
snapshots = np.empty([len(range(0, passos, frequenciaSnapshots)) * numCorpos, 8], dtype=np.float64)
contSnap = 1

In [140]:
%%time
for i in range(passos):
  if(i % frequenciaSnapshots == 0):
      inicio = contSnap * numCorpos - numCorpos
      fim = contSnap * numCorpos
      contSnap += 1
      snapshots[inicio:fim, 0:8] = np.array([x, y, z, vx, vy,
                                             vz, m, np.full(numCorpos, i)]).transpose()

  movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 622 ms, sys: 6.22 ms, total: 628 ms
Wall time: 637 ms


In [141]:
df = pd.DataFrame(
    snapshots,
    columns=['x', 'y', 'z', 'vx', 'vy', 'vz', 'm', 'passo']
)

salvarSnapshot(df, biblioteca="NumbaCPU", dimensao="D3")

In [142]:
%time kFinal = calculaEnergiaK(m, vx, vy, vz, numCorpos)

CPU times: user 13 µs, sys: 2 µs, total: 15 µs
Wall time: 16.9 µs


In [143]:
%time uFinal = calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon)

CPU times: user 15 µs, sys: 2 µs, total: 17 µs
Wall time: 18.6 µs


In [144]:
salvarHDF5(kInicial, uInicial, kFinal, uFinal, biblioteca="NumbaCPU", dimensao="D3")

In [145]:
print("Energia Mecânica Final: ", kFinal + uFinal)

Energia Mecânica Final:  -0.004427582177635496


---
## <font color="#40BCD8">Resultados</font>

---
### <font color="#33AAFF">Numba CPU</font>

---
#### <font color="#E86C4A">3D</font>

In [146]:
df = pd.read_hdf("SaidaTeste.h5", "NumbaCPU/D3/Resultados")

In [147]:
df.EnergiaCineticaInicial

,EnergiaCineticaInicial
0,0.004423


In [148]:
df.EnergiaPotencialInicial

,EnergiaPotencialInicial
0,-0.00885


In [149]:
df.EnergiaCineticaFinal

,EnergiaCineticaFinal
0,0.004321


In [150]:
df.EnergiaPotencialFinal

,EnergiaPotencialFinal
0,-0.008748
